# Neuropixels processing

End-to-end histology / Neuropixels-alignment workflow for one session. The notebook runs steps **1, 3, 5, 6 and 7**; the acquisition-time and manual steps happen outside it — **step 0** (probe geometry + spike sorting) at acquisition, **step 2** (register probe tracks to anatomy, in brainreg + napari) between steps 1 and 3, and **step 4** (the IBL ephys-alignment GUI) between steps 3 and 5:

1. **Light-sheet volume assembly** — combine raw light-sheet microscopy acquisitions into single BigTIFF volumes that brainreg / napari ingest. Two acquisition modalities are supported (LaVision UltraMicroscope, LifeCanvas SmartSPIM).
3. **IBL ephys pre-alignment export** — bridge Kilosort output + brainreg track tracing into the IBL ephys-alignment GUI's expected inputs (track points + an ALF dataset), replacing the upstream `atlaselectrophysiology.extract_files.extract_data` call, which streams the raw concatenated `.ap.bin` and takes hours on multi-hour Neuropixels 2.0 sessions even though every output is derivable from the Kilosort directory + the `.ap.meta` alone.
5. **IBL ephys post-alignment export** — post-process the GUI's per-shank channel-location JSONs into a single SpikeInterface-ready `channel_locations.json`.
6. **Channel-brain area converter** — merge this session's per-probe brain-region map (keyed by Kilosort row index) into the global channel-brain area converter, add-if-missing so every other session is preserved.
7. **Spike quality metrics** — compute the per-unit spike quality-metrics catalog (`unit_catalog.csv`) plus each probe's per-shank channel order.

The cells are organised as *(1) imports* then *(2) per-section execution*; **each section cell defines its own parameters at the top** (there is no shared parameters cell). Edit those inline values (plus `EXPERIMENTER` at the top of the imports cell). Paths are written `/mnt/...` and wrapped in `configure_path()` so they resolve on macOS (`/Volumes/...`) too.

In [ ]:
### Imports

# Auto-reload package modules whenever they change on disk, so edits to
# the usv_playpen source are picked up without restarting the kernel.
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import json
import os
from pathlib import Path

# Experimenter id. Set this BEFORE the usv_playpen imports below. The data paths
# in this notebook are written under the shipped experimenter (Bartul) and
# re-keyed to the experimenter in use by resolve_experimenter_path(). Leave
# EXPERIMENTER = None to use this machine's configured experimenter
# (_config/behavioral_experiments_settings.toml); set it to an id like
# "Annegret" to resolve every path under that experimenter's tree instead
# (restart the kernel to change it after import).
EXPERIMENTER = None
if EXPERIMENTER:
    os.environ["EXPERIMENTER_ID"] = EXPERIMENTER

from usv_playpen.os_utils import configure_path, resolve_experimenter_path
from usv_playpen.visualizations.plot_style import apply_plot_style
from usv_playpen.neuropixels.histology_stack_lightsheet_volume import stack_lightsheet_volume
from usv_playpen.neuropixels.histology_stitch_smartspim_tiles import stitch_smartspim_tiles
from usv_playpen.neuropixels.histology_ibl_alignment_export import IBLAlignmentExporter
from usv_playpen.neuropixels.anatomy_converter import add_session_to_anatomy_converter
from usv_playpen.neuropixels.spike_quality_metrics import SpikeQualityMetricsExtractor

apply_plot_style()

# Load the analyses settings once; each section cell below pulls its stable-tunable block.
with open(Path.cwd().parent / "_parameter_settings" / "analyses_settings.json", 'r') as analyses_settings_file:
    analyses_settings = json.load(analyses_settings_file)

## 1. Light-sheet volume assembly

### LaVision UltraMicroscope

1.1× objective, 5.91 µm/pix lateral, 10 µm axial. The raw acquisition is a flat directory of OME-TIFF Z-planes per channel, single tile. Use `stack_lightsheet_volume` to glue the planes into one BigTIFF per channel. Optionally flips each plane in-plane (`xy_flip` ∈ `{'none', 'vertical', 'horizontal', 'both'}`; `'both'` ≡ 180° rotation) and reverses Z order so the output is dorsal-first.

### LifeCanvas SmartSPIM

1.625× objective, 4.02 µm/pix lateral, 10 µm axial. The raw acquisition is a tiled XY grid of Z-stacks per channel under `Ex_{wavelength}_Ch{n}/{X}/{X}_{Y}/`. Use `stitch_smartspim_tiles` to read `metadata.txt`, convert stage coordinates (0.1 µm units) to pixel offsets, and stream a plane-by-plane stitch with a bevel-shaped linear feather (`feather_pixels`) over tile seams.

**Channels.** Both volume-assembly functions default to `wavelength_nm=(488, 561)` and process both channels in one call, writing one BigTIFF per channel. To restrict to a single channel, pass an `int` (e.g. `wavelength_nm=561`).

**Per-call inputs (set inline in each cell below):**
* `raw_dir` — acquisition root.
* `output_path` — destination BigTIFF path. When multiple wavelengths are requested, this **must** contain the substring `{wavelength_nm}`, which is formatted per channel (e.g. `'.../{wavelength_nm}nm.tif'` → `'.../488nm.tif'` and `'.../561nm.tif'`).
* `wavelength_nm` — defaults to both channels; pass an int to restrict.

**Stable tunables** — `xy_flip`, `z_flip`, `feather_pixels`, `skip_first` — are loaded from `_parameter_settings/analyses_settings.json`. Override any of them by passing an explicit kwarg to the function call.

**Orientation controls.** Two independent knobs, both set by trial-and-error per dataset:

* `xy_flip` flips each 2D plane *in-plane* (one of `'none'`, `'vertical'` for top↔bottom, `'horizontal'` for left↔right, `'both'` = 180° rotation). It affects the **axial** view in napari.
* `z_flip` reverses the Z (depth) iteration order before writing. It affects the **coronal** and **sagittal** views in napari (both views share the Z axis, so a wrong-handed Z manifests in both at once).

Dropped the earlier auto-detection of acquisition direction (`dv` / `vd`) because the labels written by ImSpector / the SmartSPIM acquisition app proved unreliable across our datasets — every dataset still needed manual tweaking. Picking `z_flip` per dataset is simpler and explicit.

**Troubleshooting in napari.** If the brain renders flipped:
* Wrong in **axial** view → tweak `xy_flip`.
* Right in axial, **upside-down in coronal and sagittal** → toggle `z_flip`.

In [ ]:
### 1. Light-sheet volume assembly — LaVision UltraMicroscope (stack into BigTIFF)

# LaVision UltraMicroscope light-sheet stacking
STACK_CFG = analyses_settings['npx_histology_stack_lightsheet_volume']
STACK_RAW_DIR = configure_path('/mnt/lightsheet/_rawData/LaVision/bmimica/251015_bmimica_178621-dv-lv-1_09-44-40')
STACK_OUTPUT_PATH = resolve_experimenter_path('/mnt/falkner/Bartul/histology/178621_2/registration/{wavelength_nm}nm/178621_{wavelength_nm}nm_fullsize.tif')

stack_lightsheet_volume(
    raw_dir=STACK_RAW_DIR,
    output_path=STACK_OUTPUT_PATH,
    **STACK_CFG,
)

In [ ]:
### 1. Light-sheet volume assembly — LifeCanvas SmartSPIM (stitch into BigTIFF)

# LifeCanvas SmartSPIM light-sheet stitching
STITCH_CFG = analyses_settings['npx_histology_stitch_smartspim_tiles']
STITCH_RAW_DIR = configure_path('/mnt/lightsheet/_rawData/SmartSPIM/jj9483/20251118_14_09_42_jj9483_181321_1x_vd_ss_1')
STITCH_OUTPUT_PATH = resolve_experimenter_path('/mnt/falkner/Bartul/histology/181321_1/registration/{wavelength_nm}nm/181321_{wavelength_nm}nm_stitched.tif')

stitch_smartspim_tiles(
    raw_dir=STITCH_RAW_DIR,
    output_path=STITCH_OUTPUT_PATH,
    **STITCH_CFG,
)

## IBL ephys-alignment export

Once the BigTIFF volumes above have been registered with brainreg and per-probe shank tracks have been traced in napari, the IBL ephys-alignment GUI is used to anchor every recording channel to a region in the Allen CCF. The GUI needs two kinds of inputs per session/probe/hemisphere:

* **Track points** — one JSON per shank in IBL-space (mlapdv, origin = bregma, micrometres), produced by transforming each shank's brainreg `.npy` output (apdvml CCF voxel-origin µm) through the Allen-bregma affine.
* **An ALF dataset** — the IBL canonical layout of `spikes.*`, `clusters.*`, `templates.*`, `channels.*` arrays produced from the Kilosort directory.

The upstream IBL pipeline (`atlaselectrophysiology.extract_files.extract_data`) accomplishes the second step but also streams the full concatenated `.ap.bin` to compute per-channel RMS-in-time maps (`_iblqc_*`). For our multi-hour Neuropixels 2.0 sessions the concatenated AP file is hundreds of GB and the RMS pass dominates wall-clock time — even though the GUI doesn't need its outputs (the reference IBL workflow output for this project contains no `_iblqc_*` files). Kilosort 4 also already writes the three `_phy_spikes_subset.*.npy` files that the IBL converter otherwise re-extracts from the raw binary.

`IBLAlignmentExporter` replicates the necessary IBL ALF outputs from the Kilosort directory + the SpikeGLX `.ap.meta` alone, with no raw-binary streaming, no `iblatlas` / `ibllib` / `phylib` / `spikeglx` dependencies, and a runtime measured in seconds rather than hours. Only Neuropixels 2.0 probes are supported (anything else raises `NotImplementedError`).

**Not matched on purpose:** `whitening_mat_inv.npy` differs because the reference IBL output for this session contains a stale `whitening_mat_inv.npy` (`max(abs) == 1.0`, `wm @ wmi_ref ≠ I`) that is not the inverse of the session's whitening matrix. The exporter copies the Kilosort-written `whitening_mat_inv.npy` instead, which is a proper inverse.

### Probe → hemisphere convention

Each `imecN` directory under `EPHYS/{session_date}_imecN/` corresponds to one probe insertion, and each insertion lands in one hemisphere of the brain. The notebook does not infer the hemisphere from the data — it reads a per-lab convention from `_parameter_settings/analyses_settings.json` under the key `npx_histology_ibl_alignment_export.probe_to_hemisphere`. The defaults match the convention used in this project (`imec0` → right, `imec1` → left); other labs / experimenters that wire their probes differently should edit that mapping in their checkout. The notebook then loops over every `(probe_id, hemisphere)` pair in the mapping so a two-probe session is processed in one cell rather than two.

### Pre-conditions

Before running the pre-alignment cells below, confirm that:

* **Brain segmentation and track tracing are complete.** The histology directory contains the animal/date/napari structure under `{cup_root}/histology/{mouse_id}/`, and per-shank track point clouds are saved as `{hemisphere}*.npy` files (one per shank, lexicographically sorted by filename).
* **The concatenated SpikeGLX binary and metadata exist** under `{cup_root}/EPHYS/{session_date}_{probe_id}/`. The `modify_files.concatenate_binary_files()` step in the standard preprocess pipeline already produces these with the correct `fileSizeBytes` / `fileTimeSecs` totals — there is no longer a separate concatenated-metadata step in this notebook.
* **Kilosort has run** and its output sits at `{cup_root}/EPHYS/{session_date}_{probe_id}/kilosort{kilosort_version}/`.

### Workflow at a glance

Phase | Step | What it produces
---|---|---
Pre-alignment | `write_xyz_picks` | `xyz_picks_shank{n}.json` per shank in `ibl_{H}H/`
Pre-alignment | `write_alf_outputs` | The full ALF layout in `ibl_{H}H/`
*(external)* Alignment | IBL ephys-alignment GUI | One `channel_locations_shank{n}.json` per shank in `ibl_{H}H/`
Post-alignment | `remap_channel_ids_to_raw` | Per-shank channel ids remapped from 0..95 to raw recording ids 0..n_chan−1
Post-alignment | `write_unified_channel_locations` | A single `channel_locations.json` for SpikeInterface

### 3. IBL ephys pre-alignment export

The exporter caches the parsed `.ap.meta` and resolved paths at construction time, so its four step methods can be called independently. The cell below loops over the `probe_to_hemisphere` mapping from `analyses_settings.json` and runs the two pre-alignment steps for each probe:

**`write_xyz_picks`** — for each shank, brainreg writes a track point cloud as an `.npy` of shape `(N, 3)` in Allen CCF apdvml voxel-origin micrometres. The IBL alignment GUI expects the same points in IBL mlapdv space (origin at bregma, ML positive right, AP positive anterior, DV positive dorsal). The conversion is a pure affine — column reorder (apdvml → mlapdv), translate by the Allen bregma landmark (`[5739, 5400, 332]` µm), flip the sign of the AP and DV axes. It does **not** require the Allen CCF NRRD volumes (`AllenAtlas.ccf2xyz` only uses the `BrainCoordinates` affine, never the image / annotation data), so `IBLAlignmentExporter` does not depend on `iblatlas` and avoids the ~300 MB atlas download.

**`write_alf_outputs`** — produces the IBL canonical ALF layout in `{cup_root}/histology/{mouse_id}/{session_date}/ibl_{hemisphere}H/`. Conceptually four sub-passes:

1. **Direct copies.** Required (always written by Kilosort 4 — missing source aborts the export): `params.py`, `cluster_KSLabel.tsv`, `whitening_mat{,_inv}.npy`, `channel_positions.npy → channels.localCoordinates.npy`. Optional (copied when present, skipped silently otherwise): the three `_phy_spikes_subset.*.npy` files. The optional triplet is only produced by the upstream IBL pipeline (`phylib.io.model.TemplateModel.save_spikes_subset_waveforms` streams the raw `.ap.bin` to extract 500 waveform snippets per template and writes the results back into the Kilosort directory) — Kilosort 4 itself does not write them. The IBL local alignment GUI does not load them either: `atlaselectrophysiology.load_data_local.LoaderLocal.get_data` reads only the `spikes` / `clusters` / `channels` ALF objects plus the optional `ephysTimeRms{AP,LF}` / `ephysSpectralDensityLF` files. So a fresh Kilosort 4 directory without phylib post-processing will print three `Optional file … skipping` lines and otherwise run to completion.
2. **Channel-level outputs.** `channels.rawInd.npy` is derived from `channel_map.npy`.
3. **Spike- and template-level outputs.** `spikes.{times, samples, clusters, templates, amps, depths}.npy`, `templates.{amps, waveforms, waveformsChannels}.npy`. Peak channels are found from the *whitened* template waveforms (matching phylib's `_channels`); amplitudes are scaled to volts via the unwhitened templates (`templates @ whitening_mat_inv`) using the `imAiRangeMax / imMaxInt / 80` formula for NP2.0 AP gain.
4. **Cluster-level outputs.** `clusters.{channels, amps, depths, peakToTrough, waveforms, waveformsChannels}.npy`. Supports phy-curated sessions where `spike_clusters.npy ≠ spike_templates.npy`: each cluster's whitened waveform is built as the spike-count-weighted average of the templates that contributed to it, mirroring phylib's `cluster_waveforms` for the dense non-sparse case. Empty clusters (cluster ids in `range(max+1)` that received no spikes after curation) get NaN in `peakToTrough` and `depths`, matching phylib's `nan_idx` convention.

**Intentionally skipped.** The IBL `extract_rmsmap` step — full streaming pass over the concatenated `.ap.bin` — is **not** run. The IBL alignment GUI does not require its `_iblqc_*` outputs (the reference IBL workflow output for this project contains none), and it is the dominant cost of `extract_data`.

Each `write_alf_outputs` call writes ~1.3 GB when only the required files are produced (fresh Kilosort 4 directories) or ~2.5 GB when the optional `_phy_spikes_subset.*.npy` triplet is also present. Wall-clock time is dominated by the optional 1.2 GB `_phy_spikes_subset.waveforms.npy` copy when present; without it the pre-alignment export completes in 30–50 seconds per probe.

Inputs:
* `os_cup_loc` — root mount point of the storage server (e.g. `/mnt/falkner/Bartul`, re-keyed to the experimenter in use).
* `mouse_id`, `session_date` — session identifiers; the same values are used for every probe in this session.
* `kilosort_version` — selects the `kilosort{version}/` subdirectory under each `EPHYS/{session_date}_{probe_id}/`.
* `out_subdir` (optional) — when set, the ALF outputs go into a sibling directory of `ibl_{hemisphere}H` rather than overwriting it. Useful when validating a new run against an existing reference; leave at `None` for production use.

In [ ]:
### 3. IBL ephys pre-alignment export — run for every probe in this session

PREALIGN_PROBE_TO_HEMISPHERE = analyses_settings['npx_histology_ibl_alignment_export']['probe_to_hemisphere']
PREALIGN_OS_CUP_LOC = resolve_experimenter_path('/mnt/falkner/Bartul')
PREALIGN_MOUSE_ID = '164335_0'
PREALIGN_SESSION_DATE = '20250912'
PREALIGN_KILOSORT_VERSION = '4'

for probe_id, hemisphere in PREALIGN_PROBE_TO_HEMISPHERE.items():
    print(f'--- {probe_id} ({hemisphere}H) ---')
    exporter = IBLAlignmentExporter(
        os_cup_loc=PREALIGN_OS_CUP_LOC,
        mouse_id=PREALIGN_MOUSE_ID,
        session_date=PREALIGN_SESSION_DATE,
        probe_id=probe_id,
        hemisphere=hemisphere,
        kilosort_version=PREALIGN_KILOSORT_VERSION,
        out_subdir=None,
    )
    for p in exporter.write_xyz_picks():
        print(p)
    exporter.write_alf_outputs()
    print(f'ALF outputs written to: {exporter.ephys_out_path}')

---

### 4. IBL ephys channel alignment (run the GUI here, outside the notebook)

With the contents of each `ibl_{hemisphere}H/` in place, launch the IBL ephys-alignment GUI separately (outside this notebook) and walk each shank's track through the alignment workflow. The GUI writes one `channel_locations_shank{n}.json` file per shank into the same `ibl_{hemisphere}H/` directory, keyed by per-shank channel index `channel_0` .. `channel_{m-1}` where `m` is the number of recording channels on each shank (96 for NP2.0 four-shank).

Once all per-shank JSONs exist for every probe, continue with the post-alignment cell below.

---

### 5. IBL ephys post-alignment export

Two steps, both pure JSON manipulation:

**`remap_channel_ids_to_raw`** — the IBL alignment GUI writes per-shank JSONs keyed by `channel_0` .. `channel_{m-1}`, i.e. local indices within each shank, not raw recording channel ids in 0 .. `n_chan − 1`. For multi-shank Neuropixels 2.0 probes (probe types 24, 1030, 2013, 2014, 2020, 2021) the IMRO table cached at construction time is used to re-key each per-shank JSON in place so its keys are raw channel ids. The IMRO data rows for these probes are `(channel, shank, bank, refid, elecid)`; the channel column is at index 0 and the shank column at index 1. Channels are walked in IMRO order and, for each shank, an index counter is bumped so the GUI's `channel_{0..m-1}` keys are mapped to the actual raw channel ids on that shank. For single-shank NP2.0 probes (no shank column) this method is a no-op.

**`write_unified_channel_locations`** — SpikeInterface and downstream analyses expect a single `channel_locations.json` keyed by raw channel id (`channel_<int>`) rather than one JSON per shank. This step concatenates all `channel_locations_shank*.json` files in the output directory, sorts entries by integer channel id (with any non-`channel_<int>` keys, e.g. `"origin"`, sorted to the end) and writes the combined file to `ibl_{hemisphere}H/channel_locations.json`.

In [ ]:
### 5. IBL ephys post-alignment export — run for every probe in this session

POSTALIGN_PROBE_TO_HEMISPHERE = analyses_settings['npx_histology_ibl_alignment_export']['probe_to_hemisphere']
POSTALIGN_OS_CUP_LOC = resolve_experimenter_path('/mnt/falkner/Bartul')
POSTALIGN_MOUSE_ID = '181322_2'
POSTALIGN_SESSION_DATE = '20251012'
POSTALIGN_KILOSORT_VERSION = '4'

for probe_id, hemisphere in POSTALIGN_PROBE_TO_HEMISPHERE.items():
    print(f'--- {probe_id} ({hemisphere}H) ---')
    exporter = IBLAlignmentExporter(
        os_cup_loc=POSTALIGN_OS_CUP_LOC,
        mouse_id=POSTALIGN_MOUSE_ID,
        session_date=POSTALIGN_SESSION_DATE,
        probe_id=probe_id,
        hemisphere=hemisphere,
        kilosort_version=POSTALIGN_KILOSORT_VERSION,
        out_subdir=None,
    )
    for p in exporter.remap_channel_ids_to_raw():
        print(p)
    out_path = exporter.write_unified_channel_locations()
    print(f'Unified channel_locations.json written to: {out_path}')

---

## 6. Channel-brain area converter

With the unified `channel_locations.json` in place, this step folds the session's per-probe brain-region map into the global channel-brain area converter `EPHYS/neuropixels_sites_to_anatomy_converter.json`, keyed by **Kilosort row index**. For each `(mouse, session, probe)` it joins the IBL `channel_locations.json` regions to Kilosort rows by physical `(lateral, axial)` position, compresses contiguous same-region runs into `{region: [[lo, hi], ...]}` half-open KS-row ranges, and **merges** that block into the converter — every other mouse / session / probe is preserved byte-for-byte, and the compact one-line-per-region formatting is kept. Because Kilosort row ordering is shank-major, each probe's ranges stay bounded inside one shank's KS-row block; the within-shank axial ordering is not always monotonic, so a single anatomical band on a shank may appear as two non-contiguous `[lo, hi]` intervals — set membership still resolves the right region regardless.

`add_session_to_anatomy_converter` is **add-if-missing**: a `(mouse, session, probe)` already in the converter is left untouched and nothing is written. Set `ANATOMY_FORCE = True` to re-regenerate a single existing block (e.g. after a re-alignment).

**Rebuilding the whole converter from scratch.** To rewrite *every* session at once — e.g. after changing the region-join logic — don't loop this per-session cell; use the batch entry point `regenerate_anatomy_converter`, exposed as `python -m usv_playpen.neuropixels.anatomy_converter --regenerate-all`, which rebuilds the Kilosort-row-keyed block for every `(mouse, session, probe)` triple already in the converter (add `--dry-run` to preview, or `--mouse/--session/--probe` to target a single triple). Downstream consumers (`make_behavioral_videos.find_region_by_channel`) read this converter with Kilosort row numbers, so keying it by KS row is what makes their region lookups correct.

### Pre-conditions

* The post-alignment step above has produced `ibl_{hemisphere}H/channel_locations.json` for every probe.
* The Kilosort `kilosort{version}/channel_positions.npy` exists under `EPHYS/{session_date}_{probe_id}/`.

### Inputs

* `ANATOMY_MOUSE_ID` — the tail-tagged animal id (the histology directory name).
* `ANATOMY_SESSION_ID` — the full session id used as the converter key (e.g. `20241107_114630`); its first 8 characters are the recording date that locates the Kilosort and IBL outputs.
* `ANATOMY_FORCE` — `False` (add-if-missing) by default; `True` re-regenerates an existing block.

Converter / ephys / histology paths default to the `data_roots` block of `analyses_settings.json` (resolved to the host OS via `configure_path`), so they need not be set here. The cell loops over every `(probe_id, hemisphere)` pair so a two-probe session is processed in one cell.

In [ ]:
### 6. Channel-brain area converter — run for every probe in this session

# Merge this session's KS-row-keyed regions into the global converter (add-if-missing).
# ANATOMY_SESSION_ID is the full converter key (first 8 chars = YYYYMMDD recording date). ANATOMY_FORCE=False
# is add-if-missing; True re-regenerates an existing block. Converter / ephys / histology paths default to the
# data_roots block of analyses_settings.json (resolved to the host OS), so they need not be set here.
ANATOMY_PROBE_TO_HEMISPHERE = analyses_settings['npx_histology_ibl_alignment_export']['probe_to_hemisphere']
ANATOMY_MOUSE_ID = '158112_0'
ANATOMY_SESSION_ID = '20241107_114630'
ANATOMY_FORCE = False

for probe_id, hemisphere in ANATOMY_PROBE_TO_HEMISPHERE.items():
    print(f'--- {probe_id} ({hemisphere}H) ---')
    summary = add_session_to_anatomy_converter(
        ANATOMY_MOUSE_ID,
        ANATOMY_SESSION_ID,
        probe_id,
        force=ANATOMY_FORCE,
        probe_to_hemisphere=ANATOMY_PROBE_TO_HEMISPHERE,
    )
    detail = f": {summary['reason']}" if summary['reason'] else ''
    print(f"{summary['status']}{detail} -> {summary['output']}")

---

## 7. Spike quality metrics

With the unified `channel_locations.json` in place for every probe, the final phase computes the per-unit spike quality-metrics catalog. `SpikeQualityMetricsExtractor` ports the per-session half of the `si_quality_metrics_Neuropixels2.0` workflow onto pinned stock `spikeinterface==0.104.3` — the two patches the workflow's SpikeInterface fork carried (a phy-peak-centred channel sparsity, and the somatic-classifier hijack of `get_num_positive_peaks`) are reimplemented in `npx_spikeinterface_helpers`, and the 3D monopolar source triangulation in `npx_monopolar_triangulation`.

### Reads the recording once

A multi-hour concatenated NP2.0 session on a network mount is hundreds of GB. The original workflow streamed it several times — once for `waveforms`, again for `spike_amplitudes`, again to project all spikes for PCA — taking hours per session. This pipeline reads the recording **once**, in two passes (both run by `run()`):

* **Core pass** — recording-free. The spike-train quality metrics (`num_spikes`, `firing_rate`, `firing_range`, `presence_ratio`, `isi_*`, `rp_contamination`, `synchrony`) need only the sorting and are computed by SpikeInterface on a no-waveforms analyzer.
* **Recording-dependent pass** — one sequential read. SpikeInterface computes the `waveforms` extension for a uniform per-unit random subsample of spikes — this is the single recording stream (sequential chunk reads, which a network mount handles well; per-spike random reads do not). `templates`, `noise_levels` and `principal_components` then compute off the `waveforms` extension with no further bulk read. From those:
  * the **template metrics** (`waveform_duration`, `peak_trough_ratio`, `fwhm`, `repolarization_slope`, `recovery_slope`, `exp_decay`, `spread`), the **somatic** classification and the **unit locations** are computed from the `templates` extension — the same templates the reference workflow used, so they stay faithful;
  * **`snr`** and the **PCA metrics** (`d_prime`, `isolation_distance`, `l_ratio`, `nn_hit_rate`, `nn_miss_rate`, `silhouette`) run through stock `compute_quality_metrics`;
  * the **amplitude metrics** (`amplitude_cutoff`, `amplitude_cv_*`, `amplitude_median`) and **`sd_ratio`** are computed directly from the `waveforms` extension — which is what lets `spike_amplitudes` (a second whole-recording stream) be dropped.

### What it produces

* `EPHYS/unit_catalog.csv` — one global 55-column per-unit catalog covering every session. Each probe's rows are merged in **idempotently**: rows already present for this `mouse_id` + `rec_date` + probe are dropped before the fresh rows are appended, so re-processing a session updates it in place rather than duplicating rows (the original notebook's blind-append silently duplicated rows on re-run). The `spiking_profile` column is left empty here — it is populated by the downstream FS/RS classifier.
* `EPHYS/{session_date}_{probe_id}/channel_order_per_shank.json` — per probe, each shank's channels ordered from the probe tip outward (from the IMRO table), required for the unit-distribution-along-shank figures.

### Pre-conditions

* The post-alignment step above has produced `ibl_{hemisphere}H/channel_locations.json` for every probe.
* The concatenated SpikeGLX binary + metadata and the Kilosort `kilosort{version}/` directory exist under `EPHYS/{session_date}_{probe_id}/` (the same artifacts the IBL export above consumes).
* `changepoints_info_*.json` is present in the EPHYS directory, and the per-recording-session `ephys/{probe_id}/cluster_data/*.npy` spike-time files exist — the catalog's `firing_rate` column is the median per-session firing rate across the recording sessions a unit appears in.

### Inputs

* `os_cup_loc`, `session_date`, `kilosort_version` — as above.
* `mouse_id` — the tail-tagged animal id (e.g. `158112_0`). It names the histology directory (and thus locates `channel_locations.json`) **and** is the value written into the catalog's `mouse_id` column, so the histology directory must be named with the tail-tagged id.

**Stable tunables** — `kilosort_version`, `num_channels_sparsity`, `shank_width_microns`, `shank_spacing_microns`, `phy_curated`, `somatic_classifier`, `job_kwargs`, `extension_params`, `quality_metric_params`, `template_metric_params`, `triangulation_max_distance_um` — are loaded from `_parameter_settings/analyses_settings.json` under `npx_spike_quality_metrics`, alongside the same `probe_to_hemisphere` convention used by the IBL export. `shank_spacing_microns` strips any inter-shank offset baked into the JSON's `lateral` field so the unit-location transform always sees a within-shank offset. `job_kwargs['n_jobs']` is the main throughput knob for the single recording read. `extension_params` / `quality_metric_params` / `template_metric_params` layer per-session overrides onto the module-default SpikeInterface extension / quality-metric / template-metric parameter dicts (the waveform window, ISI / refractory thresholds, subsample sizes and seeds, exp-decay R² floor, spread threshold, ...), and `triangulation_max_distance_um` caps the monopolar-triangulation source reach — every key except `probe_to_hemisphere` is forwarded straight to the extractor as a keyword argument. The cell loops over every `(probe_id, hemisphere)` pair so a two-probe session is processed in one cell.

In [ ]:
### 7. Spike quality metrics — run for every probe in this session

# Spike quality metrics
SQM_SETTINGS = analyses_settings['npx_spike_quality_metrics']
SQM_PROBE_TO_HEMISPHERE = SQM_SETTINGS['probe_to_hemisphere']
SQM_EXTRACTOR_KWARGS = {key: value for key, value in SQM_SETTINGS.items() if key != 'probe_to_hemisphere'}
SQM_OS_CUP_LOC = resolve_experimenter_path('/mnt/falkner/Bartul')
SQM_MOUSE_ID = '158112_0'
SQM_SESSION_DATE = '20241107'

for probe_id, hemisphere in SQM_PROBE_TO_HEMISPHERE.items():
    print(f'--- {probe_id} ({hemisphere}H) ---')
    extractor = SpikeQualityMetricsExtractor(
        os_cup_loc=SQM_OS_CUP_LOC,
        mouse_id=SQM_MOUSE_ID,
        session_date=SQM_SESSION_DATE,
        probe_id=probe_id,
        hemisphere=hemisphere,
        **SQM_EXTRACTOR_KWARGS,
    )
    catalog = extractor.run()
    print(f'catalog ({catalog.shape[0]} units) written to: {extractor.catalog_path}')